<a href="https://colab.research.google.com/github/ayeshayaz/python-practice/blob/main/pandas_pipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

# Load the raw csv directly
df = pd.read_csv("https://calmcode.io/static/data/bigmac.csv")

# Your data processing pipeline
df2 = (df
       .assign(date=lambda d: pd.to_datetime(d['date']))
       .sort_values(['currency_code', 'date'])
       .groupby('currency_code')
       .agg(n=('date', 'count')))

df.loc[lambda d: d['currency_code'].isin(df2[df2['n'] >= 32].index)]

,date,currency_code,name,local_price,dollar_ex,dollar_price
0,2000-04-01,ARS,Argentina,2.50,1.0000,2.500000
1,2000-04-01,AUD,Australia,2.59,1.6800,1.541667
2,2000-04-01,BRL,Brazil,2.95,1.7900,1.648045
3,2000-04-01,CAD,Canada,2.85,1.4700,1.938776
4,2000-04-01,CHF,Switzerland,5.90,1.7000,3.470588
...,...,...,...,...,...,...
1321,2020-01-14,SEK,Sweden,51.50,9.4591,5.444493
1322,2020-01-14,THB,Thailand,115.00,30.2775,3.798200
1324,2020-01-14,TWD,Taiwan,72.00,29.8845,2.409276
1327,2020-01-14,USD,United States,5.67,1.0000,5.670000


In [4]:
import pandas as pd

# 1. Load the dataset using the correct direct URL
df = pd.read_csv("https://calmcode.io/static/data/bigmac.csv")

# 2. Find how many times each currency appears in the data
df2 = (df
       .assign(date=lambda d: pd.to_datetime(d['date']))
       .sort_values(['currency_code', 'date'])
       .groupby('currency_code')
       .agg(n=('date', 'count')))

# 3. Filter original data to only keep currencies with 32 or more entries
df_filtered = df.loc[lambda d: d['currency_code'].isin(df2[df2['n'] >= 32].index)]

In [5]:
import pandas as pd

# 1. Load the dataset using the correct direct URL
df = pd.read_csv("https://calmcode.io/static/data/bigmac.csv")

# 2. Define your cleaning steps as individual functions
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

def remove_outliers(dataf):
    min_row_country = 32
    countries = (dataf
                 .groupby('currency_code')
                 .agg(n=('name', 'count'))
                 .loc[lambda d: d['n'] >= min_row_country]
                 .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

# 3. Clean and filter the data sequentially using .pipe()
cleaned_df = df.pipe(set_dtypes).pipe(remove_outliers)

In [7]:
import pandas as pd

# Load the dataset using the correct direct URL
df = pd.read_csv("https://calmcode.io/static/data/bigmac.csv")

def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

def remove_outliers(dataf, min_row_country=32):
    countries = (dataf
                .groupby('currency_code')
                .agg(n=('name', 'count'))
                .loc[lambda d: d['n'] >= min_row_country]
                .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

# Run the pipeline. Want to try a stricter threshold? Just change the argument!
cleaned_df = (df
              .pipe(set_dtypes)
              .pipe(remove_outliers, min_row_country=32))

In [8]:
import pandas as pd

df = pd.read_csv('https://calmcode.io/static/data/bigmac.csv')

def start_pipeline(dataf):
    return dataf.copy()

def set_types(dataf):
    # .assign() safely creates a new dataframe without touching the original
    return dataf.assign(date=pd.to_datetime(dataf['date']))

# Run the pipeline
pipeline_result = df.pipe(start_pipeline).pipe(set_types)

# Now check the data types:
print("Pipeline Output:")
print(pipeline_result.dtypes)

print("\nOriginal df (remains untouched!):")
print(df.dtypes)

Pipeline Output:
date             datetime64[ns]
currency_code            object
name                     object
local_price             float64
dollar_ex               float64
dollar_price            float64
dtype: object

Original df (remains untouched!):
date              object
currency_code     object
name              object
local_price      float64
dollar_ex        float64
dollar_price     float64
dtype: object


In [9]:
from functools import wraps
import datetime as dt

def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        tic = dt.datetime.now()
        result = func(*args, **kwargs)
        time_taken = str(dt.datetime.now() - tic)
        print(f"just ran step {func.__name__} shape={result.shape} took {time_taken}s")
        return result
    return wrapper

In [11]:
import time
from functools import wraps
import pandas as pd

# 1. Define the @log_step decorator
def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start_time

        # This prints the function name, resulting shape, and execution time
        print(f"[{func.__name__:<15}] shape={str(result.shape):<10} took {duration:.4f}s")
        return result
    return wrapper

# 2. Load the dataset using the correct direct URL
df = pd.read_csv('https://calmcode.io/static/data/bigmac.csv')

# 3. Define the decorated pipeline steps
@log_step
def start_pipeline(dataf):
    return dataf.copy()

@log_step
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

@log_step
def remove_outliers(dataf, min_row_country=32):
    countries = (dataf
                .groupby('currency_code')
                .agg(n=('name', 'count'))
                .loc[lambda d: d['n'] >= min_row_country]
                .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

# 4. Run the pipeline
cleaned_df = (df
              .pipe(start_pipeline)
              .pipe(set_dtypes)
              .pipe(remove_outliers, min_row_country=32))

[start_pipeline ] shape=(1330, 6)  took 0.0001s
[set_dtypes     ] shape=(1330, 6)  took 0.0050s
[remove_outliers] shape=(864, 6)   took 0.0070s


In [12]:
import time
from functools import wraps
import pandas as pd

# 1. Define the decorator to log your pipeline steps
def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start_time
        print(f"[{func.__name__:<15}] shape={str(result.shape):<10} took {duration:.4f}s")
        return result
    return wrapper

# 2. Load the dataset using the correct direct URL
df = pd.read_csv('https://calmcode.io/static/data/bigmac.csv')

# 3. Define the decorated functions
@log_step
def start_pipeline(dataf):
    return dataf.copy()

@log_step
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

@log_step
def remove_outliers(dataf, min_row_country=32):
    countries = (dataf
                .groupby('currency_code')
                .agg(n=('name', 'count'))
                .loc[lambda d: d['n'] >= min_row_country]
                .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

# 4. Run your pipeline with the lower threshold of 20!
cleaned_df = (df
              .pipe(start_pipeline)
              .pipe(set_dtypes)
              .pipe(remove_outliers, min_row_country=20))

[start_pipeline ] shape=(1330, 6)  took 0.0001s
[set_dtypes     ] shape=(1330, 6)  took 0.0055s
[remove_outliers] shape=(1248, 6)  took 0.0060s


In [13]:
import time
from functools import wraps
import pandas as pd

# 1. Define the decorator to log your pipeline steps
def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start_time
        print(f"[{func.__name__:<22}] shape={str(result.shape):<10} took {duration:.4f}s")
        return result
    return wrapper

# 2. Load the dataset using the correct direct URL
df = pd.read_csv('https://calmcode.io/static/data/bigmac.csv')

# 3. Define the decorated functions
@log_step
def start_pipeline(dataf):
    return dataf.copy()

@log_step
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

@log_step
def remove_outliers(dataf, min_row_country=32):
    countries = (dataf
                .groupby('currency_code')
                .agg(n=('name', 'count'))
                .loc[lambda d: d['n'] >= min_row_country]
                .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

@log_step
def add_inflation_features(dataf):
    return (dataf
            .assign(local_inflation=lambda d: d.groupby('name')['local_price'].pct_change())
            .assign(dollar_inflation=lambda d: d.groupby('name')['dollar_price'].pct_change()))

# 4. Run the complete pipeline
clean_df = (df
            .pipe(start_pipeline)
            .pipe(set_dtypes)
            .pipe(remove_outliers, min_row_country=20)
            .pipe(add_inflation_features))

[start_pipeline        ] shape=(1330, 6)  took 0.0001s
[set_dtypes            ] shape=(1330, 6)  took 0.0037s
[remove_outliers       ] shape=(1248, 6)  took 0.0047s
[add_inflation_features] shape=(1248, 8)  took 0.0201s


In [14]:
import pandas as pd
import altair as alt
import time
from functools import wraps

# --- DECORATOR & FUNCTIONS ---

def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start_time
        print(f"[{func.__name__:<22}] shape={str(result.shape):<10} took {duration:.4f}s")
        return result
    return wrapper

@log_step
def start_pipeline(dataf):
    return dataf.copy()

@log_step
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

@log_step
def remove_outliers(dataf, min_row_country=32):
    countries = (dataf
                .groupby('currency_code')
                .agg(n=('name', 'count'))
                .loc[lambda d: d['n'] >= min_row_country]
                .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

@log_step
def add_inflation_features(dataf):
    return (dataf
            .assign(local_inflation=lambda d: d.groupby('name')['local_price'].pct_change())
            .assign(dollar_inflation=lambda d: d.groupby('name')['dollar_price'].pct_change()))

@log_step
def drop_nans(dataf):
    # Safely drop the first row of each country containing NaNs from inflation calculations
    return dataf.dropna(subset=['local_inflation', 'dollar_inflation'])

# --- YOUR PLOT FUNCTION ---

def plot_bigmac(dataf):
    return (alt.Chart(dataf)
      .mark_point()
      .encode(x='local_inflation',
              y='dollar_inflation',
              color=alt.Color('currency_code'),
              tooltip=["currency_code", "local_inflation", "dollar_inflation"])
      .properties(width=600, height=300) # Increased height slightly to make it easier to view
      .interactive())

# --- EXECUTE THE WHOLE CHAIN ---

df = pd.read_csv('https://calmcode.io/static/data/bigmac.csv')

chart = (df
         .pipe(start_pipeline)
         .pipe(set_dtypes)
         .pipe(remove_outliers, min_row_country=20)
         .pipe(add_inflation_features)
         .pipe(drop_nans) # Added to ensure Altair has clean numeric coordinates to plot
         .pipe(plot_bigmac))

# Display the chart
chart

[start_pipeline        ] shape=(1330, 6)  took 0.0001s
[set_dtypes            ] shape=(1330, 6)  took 0.0045s
[remove_outliers       ] shape=(1248, 6)  took 0.0050s
[add_inflation_features] shape=(1248, 8)  took 0.0090s
[drop_nans             ] shape=(1206, 8)  took 0.0011s


alt.Chart(...)

In [15]:
import datetime as dt
from functools import wraps
import pandas as pd
import altair as alt

# --- Decorator ---
def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        tic = dt.datetime.now()
        result = func(*args, **kwargs)
        time_taken = str(dt.datetime.now() - tic)
        print(f"just ran step {func.__name__:<22} shape={str(result.shape):<10} took {time_taken}s")
        return result
    return wrapper

# --- Pipeline Steps ---

@log_step
def start_pipeline(dataf):
    return dataf.copy()

@log_step
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

@log_step
def remove_short_history_countries(dataf, min_row_country=20):
    # Filter out countries with too few records first
    countries = (dataf
                 .groupby('currency_code')
                 .agg(n=('name', 'count'))
                 .loc[lambda d: d['n'] >= min_row_country]
                 .index)
    return dataf.loc[lambda d: d['currency_code'].isin(countries)]

@log_step
def add_inflation_features(dataf):
    # Calculate inflation using .pct_change() (or your diff method) safely on clean countries
    return (dataf
            .assign(local_inflation=lambda d: d.groupby('name')['local_price'].pct_change())
            .assign(dollar_inflation=lambda d: d.groupby('name')['dollar_price'].pct_change()))

@log_step
def remove_extreme_inflation(dataf):
    # Now that inflation is calculated, safely strip away the NaNs and extreme values!
    return (dataf
            .dropna(subset=['local_inflation', 'dollar_inflation'])
            .loc[lambda d: d['local_inflation'] > -20])

# --- Plot Function ---
def plot_bigmac(dataf):
    return (alt.Chart(dataf)
      .mark_point()
      .encode(x='local_inflation',
              y='dollar_inflation',
              color=alt.Color('currency_code'),
              tooltip=["currency_code", "local_inflation", "dollar_inflation"])
      .properties(width=600, height=300)
      .interactive())

# --- Run everything ---
df = pd.read_csv('https://calmcode.io/static/data/bigmac.csv')

chart = (df
  .pipe(start_pipeline)
  .pipe(set_dtypes)
  .pipe(remove_short_history_countries, min_row_country=20)
  .pipe(add_inflation_features)
  .pipe(remove_extreme_inflation)
  .pipe(plot_bigmac))

chart

just ran step start_pipeline         shape=(1330, 6)  took 0:00:00.000101s
just ran step set_dtypes             shape=(1330, 6)  took 0:00:00.003786s
just ran step remove_short_history_countries shape=(1248, 6)  took 0:00:00.004859s
just ran step add_inflation_features shape=(1248, 8)  took 0:00:00.009773s
just ran step remove_extreme_inflation shape=(1206, 8)  took 0:00:00.001922s


alt.Chart(...)

In [17]:
%%writefile common.py
import pandas as pd
import datetime as dt
from functools import wraps
import altair as alt

def log_step(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        tic = dt.datetime.now()
        result = func(*args, **kwargs)
        time_taken = str(dt.datetime.now() - tic)
        print(f"just ran step {func.__name__:<22} shape={str(result.shape):<10} took {time_taken}s")
        return result
    return wrapper

@log_step
def start_pipeline(dataf):
    return dataf.copy()

@log_step
def set_dtypes(dataf):
    return (dataf
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values(['currency_code', 'date']))

@log_step
def add_inflation_features(dataf):
    return (dataf
            .assign(local_inflation=lambda d: d.groupby('name')['local_price'].pct_change())
            .assign(dollar_inflation=lambda d: d.groupby('name')['dollar_price'].pct_change()))

@log_step
def remove_outliers(dataf, min_row_country=32):
    countries = (dataf
                .groupby('currency_code')
                .agg(n=('name', 'count'))
                .loc[lambda d: d['n'] >= min_row_country]
                .index)
    return (dataf
            .loc[lambda d: d['currency_code'].isin(countries)]
            .dropna(subset=['local_inflation', 'dollar_inflation'])
            .loc[lambda d: d['local_inflation'] > -20])

def plot_bigmac(dataf):
    return (alt.Chart(dataf)
      .mark_point()
      .encode(x='local_inflation',
              y='dollar_inflation',
              color=alt.Color('currency_code'),
              tooltip=["currency_code", "local_inflation", "dollar_inflation"])
      .properties(width=600, height=300)
      .interactive())

Writing common.py
